In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed

from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV
from sklearn.tree import export_graphviz

In [4]:
arbres = pd.read_csv("train.csv", index_col = 'ID_ARBRE')
arbres_trie = arbres[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien"]]
y = arbres[["classification_diagnostic"]]

prev = pd.read_csv("prev.csv", index_col = 'ID_ARBRE')
prev_trie = prev[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien"]]

## Entrainement sur Ordinal & One-Hot-Encoder

### Ordinal & One-Hot

In [43]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Chargement des fichiers
train = pd.read_csv("train.csv", index_col = 'ID_ARBRE')
train = train[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien", "classification_diagnostic"]]

prev = pd.read_csv("prev.csv")
prev = prev[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien"]]

target = 'classification_diagnostic'

# --- 1. Définir les mappings pour les variables ordinales ---
# Ces mappings sont construits à partir des légendes fournies
ordinal_mappings = {
    "classe_age": {"J": 0, "JA": 1, "A": 2, "AM": 3},
    "classe_hauteur": {"H1": 0, "H2": 1, "H3": 2, "H4": 3, "H5": 4},
    "classe_circonference": {"C1": 0, "C2": 1, "C3": 2, "C4": 3, "C5": 4, "C6": 5, "C7": 6},
    "vigueur_pousse": {"D": 0, "PP": 1, "MP": 2, "P": 3},
    "plaie_collet" : {"AU":-1, "RCPPL":0, "RCPLC":1, "RCPLS":2, "RCPLCF":3, "RCPLNC":4, "RCPLNS":5},
    "fissure_houppier": {"HPF ": 0, "HFO ": 1, "HFF " : 2},
    "esperance_maintien": {"1.0": 1, "2.0": 2, "3.0": 3, "4.0": 4}
}

# Liste des colonnes ordinales à encoder (selon le mapping défini)
ordinal_cols = list(ordinal_mappings.keys())

# Fonction pour appliquer un mapping ordinal sur un DataFrame
def apply_ordinal_mapping(df, cols_mapping):
    for col, mapping in cols_mapping.items():
        if col in df.columns:
            df[col] = df[col].astype(str).map(mapping)
    return df

# Application sur les datasets d'entraînement et de prédiction
train = apply_ordinal_mapping(train, ordinal_mappings)
prev = apply_ordinal_mapping(prev, ordinal_mappings)

# --- 2. Encoder les variables nominales avec One-Hot Encoding ---
# On considère comme nominales toutes les colonnes (sauf la cible et les ordinales)
cols_to_exclude = ordinal_cols + [target]
nominal_cols = [col for col in train.columns if col not in cols_to_exclude]

# On applique get_dummies et on supprime la première modalité pour éviter la colinéarité (drop_first=True)
train = pd.get_dummies(train, columns=nominal_cols, drop_first=True)
prev = pd.get_dummies(prev, columns=nominal_cols, drop_first=True)

# --- 3. Aligner les colonnes de train et prev ---
# Cela permet de s'assurer que les deux DataFrames possèdent exactement les mêmes colonnes
train, prev = train.align(prev, join='left', axis=1, fill_value=0)

train.head()

,classe_age,classe_hauteur,classe_circonference,vigueur_pousse,plaie_collet,fissure_houppier,esperance_maintien,classification_diagnostic,type_sol_G,type_sol_GR,...,plaie_tronc_TPLNC,plaie_tronc_TPLS,plaie_tronc_TPPL,plaie_tronc_ZZ,bois_mort_houppier_HBMU,bois_mort_houppier_HPBM,plaie_houppier_HPLNC,plaie_houppier_HPLS,plaie_houppier_HPPL,plaie_houppier_ZZ
ID_ARBRE,,,,,,,,,,,,,,,,,,,,,
0,2,0,0,3,0,0,1,C1,False,False,...,False,True,False,False,False,True,False,True,False,False
1,2,1,2,3,2,1,2,C4,False,False,...,True,False,False,False,False,True,True,False,False,False
2,2,0,1,3,0,0,1,C1,False,True,...,False,False,False,False,False,True,False,True,False,False
3,2,1,2,3,1,0,1,C2,False,False,...,False,False,False,False,False,True,False,True,False,False
4,1,0,0,3,0,0,1,C2,False,False,...,False,True,False,False,False,True,False,False,True,False


In [11]:
import pandas as pd
import numpy as np

df = pd.DataFrame(prev)

# Affichage des valeurs uniques de chaque colonne
for col in df.columns:
    unique_values = np.unique(df[col])
    print(f'Colonne {col} : {unique_values}')


Colonne classe_age : [0 1 2]
Colonne classe_hauteur : [0 1 2 3]
Colonne classe_circonference : [0 1 2 3]
Colonne vigueur_pousse : [1 2 3]
Colonne plaie_collet : [0 1 2 4]
Colonne fissure_houppier : [0 2]
Colonne esperance_maintien : [1 2 3 4]
Colonne classification_diagnostic : [0]
Colonne type_sol_G : [False  True]
Colonne type_sol_GR : [False  True]
Colonne type_sol_Gr : [False  True]
Colonne type_sol_MA : [False  True]
Colonne type_sol_P : [False  True]
Colonne type_sol_S : [False  True]
Colonne type_sol_TV : [0]
Colonne port_arbre_L : [False  True]
Colonne port_arbre_R5 : [False  True]
Colonne port_arbre_SL : [False  True]
Colonne fissure_tronc_TFO : [0]
Colonne fissure_tronc_TPF : [False  True]
Colonne plaie_tronc_TPLCF : [0]
Colonne plaie_tronc_TPLNC : [False  True]
Colonne plaie_tronc_TPLS : [False  True]
Colonne plaie_tronc_TPPL : [False  True]
Colonne plaie_tronc_ZZ : [0]
Colonne bois_mort_houppier_HBMU  : [0]
Colonne bois_mort_houppier_HPBM  : [False  True]
Colonne plaie_houp

### Train_test_split

In [1764]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV

In [1766]:
X_train, X_test, Y_train, Y_test = train_test_split(train.drop(["classification_diagnostic"], axis=1), train['classification_diagnostic'], test_size=0.20, random_state = 42)

### Decision Tree model training

In [1769]:
tree = DecisionTreeClassifier(
    max_depth= 5,
    min_samples_leaf = 1,
    min_samples_split = 2
)

In [1771]:
tree.fit(X_train, Y_train)

DecisionTreeClassifier(max_depth=5)

In [1773]:
tree.score(X_test, Y_test)

0.8256880733944955

### Decision Trees Grid Search CV

In [1083]:
param_grid = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'ccp_alpha': [0.0, 0.01, 0.1],
    'splitter': ['best', 'random'],
    'max_leaf_nodes': [None, 10, 20, 50],
    'min_impurity_decrease': [0.0, 0.001, 0.01]
}


grid_search = GridSearchCV(
    estimator = tree,
    param_grid = param_grid,
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
    scoring = 'accuracy',
    n_jobs=-1
)

In [109]:
grid_search.fit(X_train, Y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=4, random_state=42, shuffle=True),
             estimator=DecisionTreeClassifier(max_depth=5), n_jobs=-1,
             param_grid={'ccp_alpha': [0.0, 0.01, 0.1],
                         'max_depth': [None, 5, 10, 15],
                         'max_leaf_nodes': [None, 10, 20, 50],
                         'min_impurity_decrease': [0.0, 0.001, 0.01],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'splitter': ['best', 'random']},
             scoring='accuracy')

In [113]:
grid_search.best_score_

0.8596882432891607

In [ ]:
new_model = grid_search.best_estimator_
new_model.score(X_test, Y_test)

In [119]:
prev_predict = new_model.predict(prev.drop(['classification_diagnostic'], axis=1))
prev_predict

array(['C2', 'C2', 'C2', 'C1', 'C3', 'C2', 'C1', 'C1', 'C1', 'C2', 'C4',
       'C2', 'C1', 'C2', 'C1', 'C1', 'C2', 'C2', 'C2', 'C3', 'C2', 'C3',
       'C1', 'C2', 'C1', 'C2', 'C1', 'C2', 'C2', 'C2', 'C3', 'C1', 'C2',
       'C2', 'C2', 'C2', 'C2', 'C1', 'C2', 'C2', 'C1', 'C2', 'C1', 'C2',
       'C2', 'C1', 'C1', 'C2', 'C2', 'C2', 'C1', 'C1', 'C1', 'C2', 'C2',
       'C1', 'C2', 'C1', 'C2', 'C2', 'C2', 'C1', 'C1', 'C1', 'C2', 'C2',
       'C2', 'C2', 'C2', 'C2', 'C3', 'C1', 'C2', 'C2', 'C2', 'C1', 'C2',
       'C1', 'C1', 'C1', 'C2', 'C2', 'C2', 'C2', 'C1', 'C1', 'C2', 'C2',
       'C1', 'C2', 'C1', 'C1', 'C1', 'C1', 'C2', 'C2', 'C1', 'C1', 'C2',
       'C1', 'C2', 'C1', 'C3', 'C1', 'C2', 'C1', 'C4', 'C1', 'C2', 'C1',
       'C2', 'C2', 'C2', 'C2', 'C1', 'C2', 'C1', 'C2', 'C2', 'C2', 'C2',
       'C2', 'C2', 'C1', 'C1', 'C2', 'C2', 'C1', 'C2', 'C2', 'C2', 'C1',
       'C2', 'C3', 'C2', 'C1', 'C2'], dtype=object)

### Bagging Classifier

In [609]:
from sklearn.ensemble import BaggingClassifier

In [1015]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(), 
                            bootstrap = False, 
                            n_estimators=500, 
                            max_samples=1.0, 
                            n_jobs=-1, 
                            random_state=42)

In [843]:
bag_clf.fit(X_train, Y_train)

BaggingClassifier(bootstrap=False, estimator=DecisionTreeClassifier(),
                  n_estimators=500, n_jobs=-1, random_state=42)

In [844]:
bag_clf.score(X_test, Y_test)

0.8715596330275229

In [849]:
prev_predict2 = bag_clf.predict(prev.drop('classification_diagnostic', axis=1))
values, counts = np.unique(prev_predict2, return_counts=True)
print(values)
print(counts)

['C1' 'C2' 'C3' 'C4']
[58 70  7  2]


### Bagging & RandomForest

In [852]:
bag_clf2 = BaggingClassifier(DecisionTreeClassifier(max_features="sqrt", max_leaf_nodes=5, splitter="random"), 
                            bootstrap = False, 
                            n_estimators=500, 
                            max_samples=1.0, 
                            n_jobs=-1, 
                            random_state=42)

In [854]:
bag_clf2.fit(X_train, Y_train)

BaggingClassifier(bootstrap=False,
                  estimator=DecisionTreeClassifier(max_features='sqrt',
                                                   max_leaf_nodes=5,
                                                   splitter='random'),
                  n_estimators=500, n_jobs=-1, random_state=42)

In [855]:
bag_clf2.score(X_test, Y_test)

0.6146788990825688

In [857]:
prev_predict3 = bag_clf.predict(prev.drop('classification_diagnostic', axis=1))
values, counts = np.unique(prev_predict3, return_counts=True)
print(values)
print(counts)

['C1' 'C2' 'C3' 'C4']
[58 70  7  2]


### Random Forest

In [1085]:
from sklearn.ensemble import RandomForestClassifier

In [1087]:
random_fst = RandomForestClassifier(n_estimators=500)

In [1089]:
random_fst.fit(X_train, Y_train)

RandomForestClassifier(n_estimators=500)

In [1091]:
print(random_fst.score(X_train, Y_train))
print(random_fst.score(X_test, Y_test))

0.9586206896551724
0.8440366972477065


### Random Forest sur les variables jugées importantes

In [982]:
dic = {round(score, 2): col for score, col in zip(random_fst.feature_importances_, train.columns)}
#print(round(score, 2), col)
dic

{0.06: 'vigueur_pousse',
 0.08: 'classe_hauteur',
 0.1: 'classe_circonference',
 0.16: 'plaie_collet',
 0.01: 'bois_mort_houppier_HBMU ',
 0.09: 'esperance_maintien',
 0.0: 'plaie_houppier_HPPL',
 0.04: 'plaie_tronc_TPLS',
 0.03: 'plaie_houppier_HPLS',
 0.02: 'bois_mort_houppier_HPBM ',
 0.05: 'plaie_tronc_TPLCF'}

In [984]:
important_features = [dic[i] for i in dic if i>0.07]
important_features

['classe_hauteur',
 'classe_circonference',
 'plaie_collet',
 'esperance_maintien']

In [990]:
train_important = train[important_features]
train_important.head()

,classe_hauteur,classe_circonference,plaie_collet,esperance_maintien
ID_ARBRE,,,,
0,0,0,0,1
1,1,2,2,2
2,0,1,0,1
3,1,2,1,1
4,0,0,0,1


In [1003]:
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(train_important, train['classification_diagnostic'], test_size=0.20, random_state = 42)

In [1009]:
random_fst2 = RandomForestClassifier(n_estimators=500)
random_fst2.fit(X_train2, Y_train2)

RandomForestClassifier(n_estimators=500)

In [1021]:
random_fst2.score(X_test2, Y_test2)

0.7706422018348624

### GridSearch2

In [1037]:
param_grid2 = {
    'max_depth': [None, 3, 5, 7, 10, 13, 15],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 6, 10],
    'max_leaf_nodes': [None, 10, 20, 50]
}


grid_search2 = GridSearchCV(
    estimator = random_fst,
    param_grid = param_grid2,
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
    scoring = 'accuracy',
    n_jobs=-1
)

In [1039]:
grid_search2.fit(X_train2, Y_train2)

GridSearchCV(cv=StratifiedKFold(n_splits=4, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(n_estimators=500), n_jobs=-1,
             param_grid={'max_depth': [None, 3, 5, 7, 10, 13, 15],
                         'max_leaf_nodes': [None, 10, 20, 50],
                         'min_samples_leaf': [1, 2, 6, 10],
                         'min_samples_split': [2, 4, 6, 10]},
             scoring='accuracy')

In [1047]:
grid_search2.best_params_

{'max_depth': 5,
 'max_leaf_nodes': None,
 'min_samples_leaf': 1,
 'min_samples_split': 2}

In [1049]:
new_model2 = grid_search2.best_estimator_

### Submition

In [677]:
sub = pd.DataFrame({'classification_diagnostic': prev_predict2})
sub.index.name = "ID_ARBRE"
id_arbre = prev_trie.index  # On récupère l'index chez x2
sub["ID_ARBRE"] = id_arbre  # Ajoute l'index comme colonne
sub = sub.set_index("ID_ARBRE")  # On définie 'ID_ARBRE' en tant qu'index

In [ ]:
sub.head()

In [679]:
sub.to_csv("sub_bagging2.csv")